In [ ]:
#| default_exp notebook_io


# Notebook IO

In [ ]:
#| export
import io
import base64
import json
from PIL import Image
from llm_sandbox_ui.cells import MarkdownCell, CodeCell, PromptCell#, AgentCell
from llm_sandbox_ui.app_config import PROMPT_SPLIT

In [ ]:
#| export

def reconstruct_cells_from_history(notebook_history):
    """Convert raw JavaScript cell data into rendered cell objects.
    
    Args:
        notebook_history: List of cell dicts from JavaScript with keys:
            cell_type, cell_id, source, outputs.
    
    Returns:
        List of cell objects (MarkdownCell, CodeCell, or PromptCell).
    
    Raises:
        ValueError: If cell_type is unknown.
    """    
    cells = []
    
    for cell_data in notebook_history:
        cell_type = cell_data['cell_type']
        cell_id = cell_data['cell_id']
        source = cell_data['source']
        outputs = cell_data.get('outputs')
        
        if cell_type == 'markdown':
            cell = MarkdownCell(source=source, cell_id=cell_id)
        
        elif cell_type == 'code':
            cell = CodeCell(source=source, outputs=outputs or [], cell_id=cell_id)
        
        elif cell_type == 'prompt':
            cell = PromptCell(source=source, outputs=outputs or '', cell_id=cell_id)

        #elif cell_type == 'agent':
        #    cell = AgentCell(source_prompt=source, source_code=outputs, cell_id=cell_id)
        else:
            raise ValueError(f"Unknown cell type: {cell_type}")
        
        cells.append(cell)
    
    return cells

In [ ]:
#| export

def reconstruct_ipynb_cell(cell):
    """Convert a Jupyter notebook cell dict to the appropriate cell class.
    
    Args:
        cell: Dict with 'cell_type', 'metadata', and type-specific keys.
            Markdown cells may have 'agent_cell' or 'prompt_cell' in metadata.
    
    Returns:
        MarkdownCell, CodeCell, PromptCell, AgentCell, or None if unknown type.
    """
    jup_cell_type = cell['cell_type']

    if jup_cell_type == 'markdown':
        #if 'agent_cell' in cell['metadata']:
        #    cell_type = 'agent'
        if 'prompt_cell' in cell['metadata']:
            cell_type = 'prompt'
        else:
            cell_type = 'markdown'
    else:
        cell_type = jup_cell_type
    
    
    if cell_type == 'markdown':
        return MarkdownCell.from_ipynb(cell)
    if cell_type == 'code':
        return CodeCell.from_ipynb(cell)
    elif cell_type == 'prompt':
        return PromptCell.from_ipynb(cell)
    #elif cell_type == 'agent':
    #    return AgentCell.from_ipynb(cell)
    
    else:
        return None


In [ ]:
#| export

def prepare_chat_history(cell_list):
    """
    Converts a list of notebook cells into a conversation history for the LLM.

    Args:
        cell_list (list): List of notebook cells.

    Returns:
        list: A list of message dictionaries (role/content) for the Chat API.
    """
    cell_context = []
    for cell in cell_list:
        cell_type = cell['cell_type']

        if cell_type == 'markdown':

            if is_ask_cell(cell):
                    question, answer = prep_prompt_cell(cell)
                    formatted_q = format_for_chat(question)
                    cell_context.append( {"role": "user", "content": formatted_q} )
                    formatted_a = format_for_chat(answer)
                    cell_context.append( {"role": "assistant", "content": formatted_a} )
            else:
                formatted = format_for_chat(prep_markdown_cell(cell))
                if formatted:

                    cell_context.append( {"role": "user", "content": formatted} )

        elif cell_type == 'code':

            sub_type = 'Code'
            response_role = 'user'

            formatted = format_for_chat(prep_code_cell(cell,cell_type=sub_type))
            if formatted:
                cell_context.append( {"role": "user", "content": formatted} )

            formatted = format_for_chat(prep_code_cell_output(cell,cell_type=sub_type))
            if formatted:
                cell_context.append( {"role": response_role, "content":formatted} )
    return cell_context


In [ ]:
#| export

def is_ask_cell(cell):
    """Check if a cell is a prompt cell with a response.
    
    Args:
        cell (dict): The JSON dictionary representing a cell.
        
    Returns:
        bool: True if cell contains the prompt/response separator, False otherwise.
    """
    source = cell['source']
    try:
        split_index =  source.index(PROMPT_SPLIT[2:-1])
        return True
    except:
        return False
         

In [ ]:
#| export

def prep_prompt_cell(cell):
    """
    Extracts text and embedded images from a markdown cell.

    Args:
        cell (dict): The JSON dictionary representing a markdown cell.

    Returns:
        tuple (list): Two lists containing strings (text content) and bytes (decoded image data).
    """
    source = cell['source']
    try:
        split_index =  source.index(PROMPT_SPLIT[2:-1])
        question = seperate_markdown(source[:split_index-1])
        answer = seperate_markdown(source[split_index+2:])
    except:
        question = seperate_markdown(source)
        answer = []

    formatted_question = ['## User Question Cell\n']+question
    return formatted_question, answer


In [ ]:
#| export

def seperate_markdown(markdown):
    """Separate markdown text blocks from embedded base64 images.
    
    Args:
        markdown: List of markdown content strings, potentially containing
            embedded base64 images in the format `(data:image/...;base64,...)`.
    
    Returns:
        List containing strings for text blocks and bytes for decoded images.
    """
    out_list = []
    for block in markdown:

        if "(data:image/" in block:
            base_64_text = block.split('base64,')[1]
            base_64_text = base_64_text.split(')')[0]
            output_image_bytes =  base64.b64decode(base_64_text)
            out_list.append(output_image_bytes )
 
        else:
            out_list.append(block)
    return out_list


In [ ]:
#| export

def prep_markdown_cell(markdown_cell):
    """
    Extracts text and embedded images from a markdown cell.

    Args:
        markdown_cell (dict): The JSON dictionary representing a markdown cell.

    Returns:
        list: A list containing strings (text content) and bytes (decoded image data).
    """
    output_list = ['## Markdown Cell\n']

    for block in markdown_cell['source']:
        if "(data:image/" in block:
            base_64_text = block.split('base64,')[1]
            base_64_text = base_64_text.split(')')[0]
            output_image_bytes =  base64.b64decode(base_64_text)
            output_list.append(output_image_bytes )
 
        else:
            output_list.append(block)
    return output_list
    

def format_for_chat(items):
    """
    Formats a list of mixed text and image bytes into the OpenAI/LiteLLM message structure.

    Args:
        items (list): A list containing strings or byte objects.

    Returns:
        list: A list of dictionaries with 'type' (text/image_url) keys.
    """
    formatted = []
    
    for item in items:
        if isinstance(item,str):
            if item != '':
                formatted.append( {"type": "text", "text": item})
        elif isinstance(item,bytes):
            img = Image.open(io.BytesIO(item))
            img_format = img.format.lower()
            base64_string = base64.b64encode(item).decode('utf-8')
            formatted.append( {"type": "image_url", "image_url": {"url": f"data:image/{img_format};base64,{base64_string}"}})
    
    return formatted


def prep_code_cell(code_cell,cell_type='Code'):
    """
    Extracts and labels source code from a code cell.

    Args:
        code_cell (dict): The JSON dictionary representing a code cell.
        cell_type (str, optional): Label for the cell. Defaults to 'Code'.

    Returns:
        list: A list containing the labeled header and the source code string.
    """
    output_list = [f'{cell_type} Cell\n']

    code_input = ''.join(code_cell['source'])
    output_list.append(code_input)

    return output_list


def prep_code_cell_output(code_cell,cell_type='Code'):
    """
    Extracts outputs (logs, streams, errors, images) from a code cell.

    Args:
        code_cell (dict): The JSON dictionary representing a code cell.
        cell_type (str, optional): Label for the cell. Defaults to 'Code'.

    Returns:
        list: A list containing text output, error traces, or image bytes.
    """
    if cell_type == 'Code':
        output_list = ['### Code Cell Output\n']
    else:
        output_list = []
        
    for block in code_cell['outputs']:
        if block['output_type'] in ['display_data', 'execute_result']:
            for key in  block['data'].keys():

                out_data = None
                block_data = block['data'][key]

                if key.endswith('json'):
                    try:
                        out_data = json.dumps(block_data,indent=4)

                    except Exception as e:
                        out_data = f'[Un-Serializable JSON Output: {key}]'

                elif key.startswith('image'):
                    try:
                        out_data = base64.b64decode(block_data)

                    except Exception as e:
                        out_data = f'[Un-Encodable Image Output: {key}]'

                elif key.startswith('text'):
                    if isinstance(block_data, list):
                        out_data = "".join(block_data)
                    else:
                        out_data = block_data
                else:
                    out_data = f'[Un-Renderable Output Type: {key}]'
                
                output_list.append(out_data)


        elif block['output_type'] == 'stream':
            output_list.append(''.join(block['text']))

        elif block['output_type'] == 'error' :
            output_list.append('#### Error\n')  
            output_list.append(f'evalue:{block["evalue"]}\n\n traceback:{block["traceback"]}')  


    return output_list
    